In [1]:
import copy
import functools
import glob
import pickle
import scipy
import sklearn
import warnings

warnings.filterwarnings('ignore')

import itertools
import graphviz as gr
import numpy as np
import os
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm


from matplotlib import style
from matplotlib import pyplot as plt
from pprint import pprint
from scipy import stats, special
from sklearn import datasets, mixture
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

import plotly.express as px
import plotly.figure_factory as ff

%matplotlib inline

style.use("fivethirtyeight")
pd.set_option("display.max_columns", 6)

np.random.seed(0)
pd.set_option('display.max_columns', None)

# File Locations


In [2]:
DATA_FOLDER = "data"
FIGURES_FOLDER = "figures"
OUTPUT_FOLDER = "output"

DATA_FOLDER = os.path.join("..",DATA_FOLDER)
FIGURES_FOLDER = os.path.join("..",FIGURES_FOLDER)
OUTPUT_FOLDER = os.path.join("..",OUTPUT_FOLDER)


ALL_BACKTEST_PATH = os.path.join(DATA_FOLDER,"all_backtest.csv")


### We need population level data now for logistic

In [3]:
county_features_df = pd.read_csv(os.path.join(DATA_FOLDER, "county_features.csv"))
colorado_county_features_df = county_features_df[county_features_df["STATE"]=="COLORADO"]

colorado_county_features_df["FIPS"] = colorado_county_features_df["FIPS"].astype(int)
relevant_cols = ["E_TOTPOP", "E_AGE17", "E_AGE65", "E_POV"]
colorado_county_features_df[relevant_cols] = colorado_county_features_df[relevant_cols].astype(int)

for f in relevant_cols[1:]:
    colorado_county_features_df[f+"_PERCENTAGE"] = 100*colorado_county_features_df[f]/colorado_county_features_df["E_TOTPOP"]

features_to_keep = ["FIPS", "COUNTY", "STATE", "LAT", "LON", "AREA_SQMI", "E_TOTPOP", "E_AGE17", "E_AGE65", "E_POV"] + [f+"_PERCENTAGE" for f in relevant_cols[1:]]

colorado_county_features_df = colorado_county_features_df[features_to_keep]

colorado_county_features_df["E_TOTPOP_RANK"] = colorado_county_features_df["E_TOTPOP"].rank(ascending=False)

colorado_county_features_df

,FIPS,COUNTY,STATE,LAT,LON,AREA_SQMI,E_TOTPOP,E_AGE17,E_AGE65,E_POV,E_AGE17_PERCENTAGE,E_AGE65_PERCENTAGE,E_POV_PERCENTAGE,E_TOTPOP_RANK
244,8001,Adams,COLORADO,39.874325,-104.331872,1166.256942,497115,135444,49181,56588,27.246009,9.893284,11.383282,5.0
245,8003,Alamosa,COLORADO,37.568442,-105.788041,722.576715,16444,3932,2113,3635,23.911457,12.849672,22.105327,31.0
246,8005,Arapahoe,COLORADO,39.644554,-104.331706,797.930917,636671,153460,78627,56802,24.103501,12.349707,8.921719,3.0
247,8007,Archuleta,COLORADO,37.202395,-107.050863,1350.086257,12908,2209,3168,1370,17.113418,24.542919,10.613573,35.0
248,8009,Baca,COLORADO,37.309780,-102.543741,2554.991586,3563,739,923,661,20.740949,25.905136,18.551782,56.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,8117,Summit,COLORADO,39.621023,-106.137555,608.338717,30429,5031,3601,3075,16.533570,11.834106,10.105491,18.0
304,8119,Teller,COLORADO,38.869976,-105.187365,557.042574,24113,4368,4875,1980,18.114710,20.217310,8.211338,24.0
305,8121,Washington,COLORADO,39.965790,-103.209744,2518.082648,4840,1108,944,480,22.892562,19.504132,9.917355,51.0
306,8123,Weld,COLORADO,40.555961,-104.383666,3986.025890,295123,78590,34304,30574,26.629575,11.623628,10.359748,9.0


### Get the actual cases on the day itself `x` from 7 days later `y` data

In [4]:
all_backtest_df = pd.read_csv(ALL_BACKTEST_PATH, parse_dates=True)
# Merge to get the TOTPOP
all_backtest_df = pd.merge(left=all_backtest_df, right=colorado_county_features_df[["FIPS","E_TOTPOP"]],how="left",left_on="fips",right_on="FIPS")

# For this analysis:
# x = 7 days ago
# y = now
# z = 7 days later
all_backtest_df = all_backtest_df.dropna(subset=["log_rolled_cases.y"])
# Generate dates
all_backtest_df["date.y"] = pd.to_datetime(all_backtest_df["date.y"])
all_backtest_df["date.x"] = all_backtest_df["date.y"] +pd.DateOffset(days=-6)
all_backtest_df["date.z"] = all_backtest_df["date.y"] +pd.DateOffset(days=6)
# Generate days from start 
all_backtest_df["days_from_start.x"] = all_backtest_df["days_from_start.y"] - 7
all_backtest_df["days_from_start.z"] = all_backtest_df["days_from_start.y"] + 7
# Sort by date within each county
all_backtest_df = all_backtest_df.groupby("fips").apply(lambda x: x.sort_values(by="date.y"))
all_backtest_df = all_backtest_df.reset_index(drop=True)
# Generate Case data for each county
all_backtest_df["log_rolled_cases.x"] = all_backtest_df.groupby("fips")["log_rolled_cases.y"].shift(7)
all_backtest_df["log_rolled_cases.z"] = all_backtest_df.groupby("fips")["log_rolled_cases.y"].shift(-7)

# Obtain ground truth growth rates one week later
all_backtest_df["r.gt.z"] = all_backtest_df["r.gt.y"].shift(-7)
# Obtain current and one week later rate prediction
all_backtest_df["r.grf.x"] = all_backtest_df["r.grf"]
all_backtest_df["r.grf.y"] = all_backtest_df.groupby("fips")["r.grf"].shift(-7)
all_backtest_df["r.grf.z"] = all_backtest_df.groupby("fips")["r.grf"].shift(-14)

# Drop NA
all_backtest_df = all_backtest_df.dropna(subset=["r.gt.y"])
all_backtest_df = all_backtest_df.dropna(subset=["r.gt.z"])
all_backtest_df = all_backtest_df.dropna(subset=["r.grf.y"])

# Obtain states so that we can rank counties within state for each county
county_fips_master_df = pd.read_csv(os.path.join(OUTPUT_FOLDER,"county_fips_master.csv"))
all_backtest_df = pd.merge(left=all_backtest_df, right = county_fips_master_df[["fips","state_abbr"]],how="left",left_on="fips",right_on="fips")
# RANKINGS should only be of eligible decision space!


In [5]:
all_backtest_df

,fips,county,r.lm,predicted.lm,date.y,days_from_start.y,log_rolled_cases.y,r.grf,tau.variance,tau.upr,tau.lwr,predicted.grf,date_delta,r.gt.y,FIPS,E_TOTPOP,date.x,date.z,days_from_start.x,days_from_start.z,log_rolled_cases.x,log_rolled_cases.z,r.gt.z,r.grf.x,r.grf.y,r.grf.z,state_abbr
0,10001,Kent,0.210565,4.457107,2020-04-04,74,3.941582,0.221232,0.000165,0.246373,0.196091,4.531776,7.0,0.138445,NaN,NaN,2020-03-29,2020-04-10,67,81,NaN,5.403803,0.212927,0.221232,0.173434,0.111785,DE
1,10001,Kent,0.130362,4.026048,2020-04-05,75,4.139159,0.165391,0.000147,0.189187,0.141595,4.271254,8.0,0.155766,NaN,NaN,2020-03-30,2020-04-11,68,82,3.113515,5.484797,0.176907,0.165391,0.187661,0.086951,DE
2,10001,Kent,0.085158,3.794778,2020-04-06,76,4.430817,0.132485,0.000105,0.152608,0.112363,4.126070,9.0,0.179872,NaN,NaN,2020-03-31,2020-04-12,69,83,3.198673,5.556828,0.142212,0.132485,0.226007,0.074701,DE
3,10001,Kent,0.124563,4.195175,2020-04-07,77,4.675163,0.150990,0.000112,0.171697,0.130283,4.380167,10.0,0.203947,NaN,NaN,2020-04-01,2020-04-13,70,84,3.323236,5.643679,0.111514,0.150990,0.217570,0.078511,DE
4,10001,Kent,0.134657,4.400491,2020-04-08,78,4.945207,0.148764,0.000148,0.172625,0.124904,4.499244,11.0,0.227909,NaN,NaN,2020-04-02,2020-04-14,71,85,3.457893,5.715382,0.092487,0.148764,0.217407,0.069231,DE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1262571,99999,New York City,-0.036250,8.298952,2021-06-28,524,8.435766,-0.038238,0.000041,-0.025628,-0.050848,8.285041,457.0,-0.013409,NaN,NaN,2021-06-22,2021-07-04,517,531,8.552705,8.313791,-0.017119,-0.038238,-0.020764,NaN,NaN
1262572,99999,New York City,-0.028884,8.321634,2021-06-29,525,8.415935,-0.034698,0.000032,-0.023567,-0.045829,8.280936,458.0,-0.012480,NaN,NaN,2021-06-23,2021-07-05,518,532,8.523821,8.297606,-0.017809,-0.034698,-0.025573,NaN,NaN
1262573,99999,New York City,-0.027698,8.302234,2021-06-30,526,8.398466,-0.036259,0.000355,0.000649,-0.073168,8.242306,459.0,-0.013526,NaN,NaN,2021-06-24,2021-07-06,519,533,8.496123,8.298104,-0.016716,-0.036259,-0.022264,NaN,NaN
1262574,99999,New York City,-0.016528,8.363902,2021-07-01,527,8.389133,-0.019052,0.000070,-0.002705,-0.035398,8.346233,460.0,-0.014445,NaN,NaN,2021-06-25,2021-07-07,520,534,8.479595,8.317705,-0.011271,-0.019052,-0.020378,NaN,NaN
